# Data preparation

Load IBTrACS with UTC timestamps, remove storms that are too short for modeling, and order each track chronologically.

In [ ]:
from pathlib import Path
import tomllib

from storm_tracks.data import load_ibtracs, prepare_tracks

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
with (PROJECT_ROOT / "configs" / "pipeline.toml").open("rb") as file:
    config = tomllib.load(file)
raw_path = PROJECT_ROOT / config["paths"]["raw_data"]
intermediate_path = PROJECT_ROOT / "data" / "hurricane_paths_prepared.pkl"

## Load the source tracks

In [ ]:
raw_tracks = load_ibtracs(raw_path)
assert str(raw_tracks["iso_time"].dtype) == "datetime64[ns, UTC]"
raw_tracks[["sid", "iso_time", "lat", "lon"]].head()

## Prepare complete trajectories

In [ ]:
prepared_tracks = prepare_tracks(
    raw_tracks,
    min_observations=config["preparation"]["min_observations"],
)
assert prepared_tracks.groupby("sid")["iso_time"].apply(
    lambda values: values.is_monotonic_increasing
).all()
intermediate_path.parent.mkdir(parents=True, exist_ok=True)
prepared_tracks.to_pickle(intermediate_path)
print(f"Saved the granular handoff to {intermediate_path}")

## Saved data

The prepared tracks are saved as the input to feature engineering. `scripts/prepare_data.py` runs preparation and feature engineering together when a fresh processed dataset is needed.